# RAG 기초

## 학습 목표

- 이 저장소에서 검색 증강 생성(retrieval-augmented generation, RAG)이 어떤 의미로 쓰이는지 이해한다.
- 문서 청크 분할(chunking)과 로컬 검색(retrieval)이 어떻게 이어지는지 단계별로 확인한다.
- 기본 TF-IDF retriever와 선택적 FAISS retriever 경로를 비교해본다.
- 단순한 RAG가 어디까지 잘 작동하고, 어디서부터 한계가 드러나는지 관찰한다.


## 개념 설명

실험을 시작하기 전에 지금 노트북이 어떤 Python 실행 환경을 쓰는지 먼저 확인한다. 로컬 노트북이든 원격 Jupyter 서버든 DGX 환경이든, 결국 이 노트북이 `uv` 환경을 정확히 바라봐야 같은 의존성과 같은 결과를 재현할 수 있다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import RuntimeConfig

print(sys.executable)
print(RuntimeConfig.auto_detect())

## RAG란 무엇인가

RAG는 검색(retrieval)과 답변 생성(generation)을 결합한 방식이다. 이 프로젝트에서는 retriever가 먼저 관련 문서 청크를 찾고, baseline answer builder가 그 근거를 이어 붙여 답을 만든다. 핵심은 단순하다. 모델이 원래 다 알고 있다고 가정하지 말고, 먼저 찾은 근거를 바탕으로 답하게 만드는 것이다.


In [ ]:
import pandas as pd

from src.ingestion import load_documents

documents = load_documents()
document_frame = pd.DataFrame(
    [
        {'doc_id': item['doc_id'], 'source': item['source'], 'characters': len(item['text'])}
        for item in documents
    ]
)
document_frame

## 문서 청크 분할(document chunking)이란 무엇인가

retriever는 보통 문서 전체보다 작은 청크 단위에서 더 잘 동작한다. 청크가 작을수록 근거 창(evidence window)이 좁아지고, 왜 특정 결과가 상위에 올랐는지 사람이 해석하기도 쉬워진다. 이 저장소는 문장 기반 청크 분할을 사용해서, 검색 결과를 그대로 읽어도 이해가 되도록 설계했다.


In [ ]:
from src.ingestion import ingest_documents

chunks = ingest_documents(persist=False)
chunk_frame = pd.DataFrame(chunks)
chunk_frame[['doc_id', 'chunk_id', 'source', 'text']].head(10)

## 임베딩 검색(embedding retrieval)이란 무엇인가

기본 retriever는 재현성과 경량성을 위해 TF-IDF와 lexical overlap을 섞은 hybrid 방식을 사용한다. 동시에 이 저장소에는 dense retrieval 실험을 위한 선택적 FAISS + sentence-transformers 경로도 들어 있다. 아래 셀에서는 두 경로를 모두 시도해보고, 현재 환경에서 어떤 retriever가 실제로 사용 가능한지 확인한다.


In [ ]:
from src.ingestion import build_demo_index

tfidf_retriever = build_demo_index(persist=False, backend='tfidf')
faiss_retriever = build_demo_index(persist=False, backend='faiss')
retriever_summary = pd.DataFrame(
    [
        {'backend': 'tfidf', 'class_name': type(tfidf_retriever).__name__},
        {'backend': 'faiss', 'class_name': type(faiss_retriever).__name__},
    ]
)
retriever_summary

## 구현

인덱스가 준비되면 검색은 결국 점수 기반 랭킹 문제가 된다. 여기서는 항상 사용 가능한 TF-IDF retriever를 먼저 확인하고, dense retrieval 의존성이 설치된 환경이라면 FAISS 경로까지 같이 살펴본다.


In [ ]:
retrieval_query = 'What are the goals of the workspace policy refresh?'
tfidf_results = pd.DataFrame(tfidf_retriever.search(retrieval_query, top_k=4))
faiss_results = pd.DataFrame(faiss_retriever.search(retrieval_query, top_k=4))

print('TF-IDF top results')
display(tfidf_results[['chunk_id', 'source', 'score', 'text']])
print('FAISS path top results')
display(faiss_results[['chunk_id', 'source', 'score', 'text']] if not faiss_results.empty else faiss_results)

## 단순 질의응답(Simple QA)

baseline RAG 경로는 계획(planning), 도구 사용(tool use), 근거 검증(grounding verification) 없이 검색한 문맥만으로 짧은 답을 만든다. 그래서 교육용 baseline으로 좋다. 검색만으로 충분한 문제와, 검색만으로는 부족한 문제를 아주 선명하게 비교할 수 있기 때문이다.


In [ ]:
from src.workflow import run_baseline_rag

baseline_result = run_baseline_rag(
    'What are the main goals of the workspace policy refresh?',
    retriever=tfidf_retriever,
)

pd.Series(
    {
        'final_status': baseline_result['final_status'],
        'final_answer': baseline_result['final_answer'],
        'retrieved_sources': [item['source'] for item in baseline_result['retrieved_docs']],
    }
)

## 실험

가장 좋은 비교 실험은 쉬운 lookup 질문 하나와, 실제로는 계획(planning)이나 도구 사용(tool use)이 필요한 질문 하나를 나란히 보는 것이다. 이렇게 보면 baseline RAG가 충분한 지점과 agentic workflow가 개입해야 하는 지점이 분명해진다.


In [ ]:
experiment_questions = [
    'When does the organization-wide rollout begin?',
    'How many days are in the pilot window?',
]
experiment_rows = []
for question in experiment_questions:
    result = run_baseline_rag(question, retriever=tfidf_retriever)
    experiment_rows.append(
        {
            'question': question,
            'final_status': result['final_status'],
            'retrieved_sources': ', '.join(sorted({doc['source'] for doc in result['retrieved_docs']})),
            'answer': result['final_answer'],
        }
    )

pd.DataFrame(experiment_rows)

## 결과 해석

아래 표는 실험 결과를 읽는 기준을 정리한 것이다. rollout date처럼 한 문서 안에 직접 답이 있는 질문은 baseline도 잘 처리한다. 반면 pilot window처럼 날짜 계산이 필요한 질문은 검색된 문장만으로는 한계가 있고, 이 지점에서 agentic workflow의 가치가 드러난다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {
            'backend': 'TF-IDF baseline',
            'strength': 'Fast, deterministic, easy to inspect',
            'limitation': 'No planning, tool use, or abstention logic',
        },
        {
            'backend': 'Optional FAISS path',
            'strength': 'Dense similarity can surface semantically close evidence',
            'limitation': 'Needs extra dependencies and more setup',
        },
    ]
)
analysis_frame

## 핵심 정리

- 이 실험을 통해 baseline RAG는 **가까운 근거를 찾고 그대로 답하는 문제**에는 충분히 강하다는 점을 확인했다.
- 동시에 **계획(planning), 도구 사용(tool use), 근거 검증(grounding verification)** 이 필요한 질문에서는 구조적 한계가 분명했다.
- dense retrieval은 검색 품질을 보완할 수 있지만, 아키텍처 자체를 대체하지는 못한다.
- 면접에서는 "왜 baseline을 먼저 두었는가"라는 질문에 대해, **후속 agentic workflow의 개선 효과를 비교하기 위한 통제군(control)** 이라고 설명하면 좋다.
